In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Load the saved model and tokenizer
model_path = "./models/financial-ner-extended/checkpoint-400"
extended_tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
extended_model = AutoModelForTokenClassification.from_pretrained(model_path, local_files_only=True)

# Move the model to the appropriate device (e.g., CPU, GPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
extended_model.to(device)

# The id2label dictionary is needed to convert predictions back to entity names
id2label = {
    0: 'O', 1: 'PER_B', 2: 'PER_I', 3: 'LOC_B', 4: 'LOC_I', 5: 'ORG_B', 6: 'ORG_I',
    7: 'AMOUNT_B', 8: 'AMOUNT_I', 9: 'DATE_B', 10: 'DATE_I',
    11: 'ACCOUNT_B', 12: 'ACCOUNT_I', 13: 'SSN_B', 14: 'SSN_I',
    15: 'FORM_B', 16: 'FORM_I'
}

In [6]:
# Test sentence
test_sentence = "The taxpayer's Social Security Number 123-45-6789 is required for Form 1040."

# Tokenize (do NOT split manually)
inputs = extended_tokenizer(test_sentence, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = extended_model(**inputs)
    predicted_label_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

# Convert IDs to labels
tokens = extended_tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
pred_labels = [id2label[i] for i in predicted_label_ids]

# Reconstruct entities using our robust function
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

entities = reconstruct_entities(tokens, pred_labels)

# Optional post-processing for ACCOUNT vs FORM
def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

entities = postprocess_entities(entities)

print("Tokens:", tokens)
print("Predicted labels:", pred_labels)
print("Detected Entities:", entities)


Tokens: ['[CLS]', 'the', 'taxpayer', "'", 's', 'social', 'security', 'number', '123', '-', '45', '-', '67', '##8', '##9', 'is', 'required', 'for', 'form', '104', '##0', '.', '[SEP]']
Predicted labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'SSN_B', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'O', 'O', 'O', 'FORM_B', 'FORM_I', 'FORM_I', 'O', 'AMOUNT_I']
Detected Entities: [('123 - 45 - 6789', 'SSN'), ('form 1040', 'FORM')]


In [7]:
# Test sentence
test_sentence = "Microsoft Corporation reported quarterly earnings to the SEC."

# Tokenize (do NOT split manually)
inputs = extended_tokenizer(test_sentence, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = extended_model(**inputs)
    predicted_label_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

# Convert IDs to labels
tokens = extended_tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
pred_labels = [id2label[i] for i in predicted_label_ids]

# Reconstruct entities using our robust function
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

entities = reconstruct_entities(tokens, pred_labels)

# Optional post-processing for ACCOUNT vs FORM
def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

entities = postprocess_entities(entities)

print("Tokens:", tokens)
print("Predicted labels:", pred_labels)
print("Detected Entities:", entities)


Tokens: ['[CLS]', 'microsoft', 'corporation', 'reported', 'quarterly', 'earnings', 'to', 'the', 'sec', '.', '[SEP]']
Predicted labels: ['O', 'ORG_B', 'ORG_I', 'O', 'O', 'O', 'O', 'O', 'ORG_B', 'O', 'AMOUNT_I']
Detected Entities: [('microsoft corporation', 'ORG'), ('sec', 'ORG')]


In [8]:
# Test sentence
test_sentence = "The taxpayer's Social Security Number is 123-45-6789."

# Tokenize (do NOT split manually)
inputs = extended_tokenizer(test_sentence, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = extended_model(**inputs)
    predicted_label_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

# Convert IDs to labels
tokens = extended_tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
pred_labels = [id2label[i] for i in predicted_label_ids]

# Reconstruct entities using our robust function
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

entities = reconstruct_entities(tokens, pred_labels)

# Optional post-processing for ACCOUNT vs FORM
def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

entities = postprocess_entities(entities)

print("Tokens:", tokens)
print("Predicted labels:", pred_labels)
print("Detected Entities:", entities)

Tokens: ['[CLS]', 'the', 'taxpayer', "'", 's', 'social', 'security', 'number', 'is', '123', '-', '45', '-', '67', '##8', '##9', '.', '[SEP]']
Predicted labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'SSN_B', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'O', 'O']
Detected Entities: [('123 - 45 - 6789', 'SSN')]


In [9]:
# Test sentence
test_sentence = "Form 1040 must be filed by April 15th for tax year 2023."

# Tokenize (do NOT split manually)
inputs = extended_tokenizer(test_sentence, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run inference
with torch.no_grad():
    outputs = extended_model(**inputs)
    predicted_label_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

# Convert IDs to labels
tokens = extended_tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
pred_labels = [id2label[i] for i in predicted_label_ids]

# Reconstruct entities using our robust function
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

entities = reconstruct_entities(tokens, pred_labels)

# Optional post-processing for ACCOUNT vs FORM
def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

entities = postprocess_entities(entities)

print("Tokens:", tokens)
print("Predicted labels:", pred_labels)
print("Detected Entities:", entities)

Tokens: ['[CLS]', 'form', '104', '##0', 'must', 'be', 'filed', 'by', 'april', '15th', 'for', 'tax', 'year', '202', '##3', '.', '[SEP]']
Predicted labels: ['O', 'FORM_B', 'FORM_I', 'FORM_I', 'O', 'O', 'O', 'O', 'DATE_B', 'DATE_I', 'O', 'O', 'O', 'DATE_B', 'DATE_I', 'O', 'DATE_I']
Detected Entities: [('form 1040', 'FORM'), ('april 15th', 'DATE'), ('2023', 'DATE')]


In [10]:
def testing(sentence):
    # Tokenize (do NOT split manually)
    inputs = extended_tokenizer(test_sentence, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Run inference
    with torch.no_grad():
        outputs = extended_model(**inputs)
        predicted_label_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

    # Convert IDs to labels
    tokens = extended_tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
    pred_labels = [id2label[i] for i in predicted_label_ids]

    # Reconstruct entities using our robust function
    def reconstruct_entities(tokens, labels):
        entities = []
        current_entity = []
        current_label = None
        for token, label in zip(tokens, labels):
            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                continue
            if token.startswith("##"):
                if current_entity:
                    current_entity[-1] += token[2:]
            elif label.endswith("_B"):
                if current_entity:
                    entities.append((" ".join(current_entity), current_label))
                current_entity = [token]
                current_label = label.split("_")[0]
            elif label.endswith("_I") and current_label == label.split("_")[0]:
                current_entity.append(token)
            else:
                if current_entity:
                    entities.append((" ".join(current_entity), current_label))
                current_entity = []
                current_label = None
        if current_entity:
            entities.append((" ".join(current_entity), current_label))
        return entities

    entities = reconstruct_entities(tokens, pred_labels)

    # Optional post-processing for ACCOUNT vs FORM
    def postprocess_entities(entities):
        cleaned = []
        for text, label in entities:
            if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
                cleaned.append((text, "ACCOUNT"))
            else:
                cleaned.append((text, label))
        return cleaned

    entities = postprocess_entities(entities)

    print("Tokens:", tokens)
    print("Predicted labels:", pred_labels)
    print("Detected Entities:", entities)
    return entities

In [11]:
# %%
# ===============================
# FINANCIAL NER INFERENCE (MPS)
# ===============================

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# -------------------------------
# Load the model and tokenizer
# -------------------------------
model_path = "./models/financial-ner-extended/checkpoint-400"
device = torch.device("mps")  # Force MPS

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(model_path, local_files_only=True)
model.to(device)
model.eval()

# Extended label mapping
id2label = {
    0: 'O', 1: 'PER_B', 2: 'PER_I', 3: 'LOC_B', 4: 'LOC_I', 5: 'ORG_B', 6: 'ORG_I',
    7: 'AMOUNT_B', 8: 'AMOUNT_I', 9: 'DATE_B', 10: 'DATE_I',
    11: 'ACCOUNT_B', 12: 'ACCOUNT_I', 13: 'SSN_B', 14: 'SSN_I',
    15: 'FORM_B', 16: 'FORM_I'
}

# -------------------------------
# Test sentence with all labels
# -------------------------------
test_sentence = (
    "John Smith from New York deposited $5,000 into account 123456789 "
    "at Bank of America on March 15, 2024. His Social Security Number "
    "123-45-6789 was used to file Form 1040."
)

# -------------------------------
# Tokenize and run inference
# -------------------------------
inputs = tokenizer(test_sentence, return_tensors="pt", truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    predicted_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
pred_labels = [id2label[i] for i in predicted_ids]

# -------------------------------
# Reconstruct entities from subwords
# -------------------------------
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

entities = reconstruct_entities(tokens, pred_labels)

# -------------------------------
# Postprocess ACCOUNT vs FORM confusion
# -------------------------------
def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        # Detect long digit sequences likely to be ACCOUNT numbers
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

entities = postprocess_entities(entities)

# -------------------------------
# Merge consecutive entities (DATE, AMOUNT, SSN, ACCOUNT, FORM)
# -------------------------------
def merge_consecutive_entities(entities):
    merged = []
    prev_text, prev_label = "", ""
    
    for text, label in entities:
        if label == prev_label and label in ["DATE", "AMOUNT", "SSN", "ACCOUNT", "FORM"]:
            prev_text += " " + text
        else:
            if prev_text:
                merged.append((prev_text, prev_label))
            prev_text, prev_label = text, label
    if prev_text:
        merged.append((prev_text, prev_label))
    return merged

entities = merge_consecutive_entities(entities)

# -------------------------------
# Print final results
# -------------------------------
print("Test Sentence:", test_sentence)
print("Tokens:", tokens)
print("Predicted labels:", pred_labels)
print("Detected Entities:", entities)


Test Sentence: John Smith from New York deposited $5,000 into account 123456789 at Bank of America on March 15, 2024. His Social Security Number 123-45-6789 was used to file Form 1040.
Tokens: ['[CLS]', 'john', 'smith', 'from', 'new', 'york', 'deposited', '$', '5', ',', '000', 'into', 'account', '123', '##45', '##6', '##7', '##8', '##9', 'at', 'bank', 'of', 'america', 'on', 'march', '15', ',', '202', '##4', '.', 'his', 'social', 'security', 'number', '123', '-', '45', '-', '67', '##8', '##9', 'was', 'used', 'to', 'file', 'form', '104', '##0', '.', '[SEP]']
Predicted labels: ['O', 'PER_B', 'PER_I', 'O', 'ORG_B', 'ORG_I', 'O', 'AMOUNT_B', 'AMOUNT_I', 'AMOUNT_I', 'AMOUNT_I', 'O', 'O', 'FORM_B', 'AMOUNT_I', 'AMOUNT_I', 'AMOUNT_I', 'AMOUNT_I', 'AMOUNT_I', 'O', 'ORG_B', 'ORG_I', 'ORG_I', 'O', 'DATE_B', 'DATE_I', 'DATE_I', 'DATE_I', 'DATE_I', 'O', 'O', 'O', 'O', 'O', 'SSN_B', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'SSN_I', 'O', 'O', 'O', 'O', 'FORM_B', 'FORM_I', 'FORM_I', 'O', 'AMOUNT_I

In [12]:
# %%
# ===============================
# FINANCIAL NER BATCH INFERENCE (MPS)
# ===============================

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from tabulate import tabulate  # For nice table output

# -------------------------------
# Load model and tokenizer
# -------------------------------
model_path = "./models/financial-ner-extended/checkpoint-400"
device = torch.device("mps")  # Force MPS

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(model_path, local_files_only=True)
model.to(device)
model.eval()

id2label = {
    0: 'O', 1: 'PER_B', 2: 'PER_I', 3: 'LOC_B', 4: 'LOC_I', 5: 'ORG_B', 6: 'ORG_I',
    7: 'AMOUNT_B', 8: 'AMOUNT_I', 9: 'DATE_B', 10: 'DATE_I',
    11: 'ACCOUNT_B', 12: 'ACCOUNT_I', 13: 'SSN_B', 14: 'SSN_I',
    15: 'FORM_B', 16: 'FORM_I'
}

# -------------------------------
# Functions for entity reconstruction
# -------------------------------
def reconstruct_entities(tokens, labels):
    entities = []
    current_entity = []
    current_label = None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if token.startswith("##"):
            if current_entity:
                current_entity[-1] += token[2:]
        elif label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_label == label.split("_")[0]:
            current_entity.append(token)
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = []
            current_label = None
    if current_entity:
        entities.append((" ".join(current_entity), current_label))
    return entities

def postprocess_entities(entities):
    cleaned = []
    for text, label in entities:
        if label == "FORM" and text.replace("-", "").isdigit() and len(text.replace("-", "")) >= 6:
            cleaned.append((text, "ACCOUNT"))
        else:
            cleaned.append((text, label))
    return cleaned

def merge_consecutive_entities(entities):
    merged = []
    prev_text, prev_label = "", ""
    for text, label in entities:
        if label == prev_label and label in ["DATE", "AMOUNT", "SSN", "ACCOUNT", "FORM"]:
            prev_text += " " + text
        else:
            if prev_text:
                merged.append((prev_text, prev_label))
            prev_text, prev_label = text, label
    if prev_text:
        merged.append((prev_text, prev_label))
    return merged

# -------------------------------
# Batch inference function
# -------------------------------
def batch_ner(sentences):
    results = []
    for sentence in sentences:
        # Tokenize and move to device
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_ids = outputs.logits.argmax(-1).squeeze().cpu().numpy()

        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
        pred_labels = [id2label[i] for i in predicted_ids]

        # Reconstruct entities
        entities = reconstruct_entities(tokens, pred_labels)
        entities = postprocess_entities(entities)
        entities = merge_consecutive_entities(entities)

        results.append((sentence, entities))
    return results

# -------------------------------
# Example batch sentences
# -------------------------------
test_sentences = [
    # Standard case with all entities
    "John Smith from New York deposited $5,000 into account 123456789 at Bank of America on March 15, 2024. His Social Security Number 123-45-6789 was used to file Form 1040.",
    
    # Multiple people and organizations
    "Alice Johnson and Bob Lee joined Microsoft Corporation and Google LLC respectively on January 2, 2023.",
    
    # Odd dollar formats
    "The company received payments of $1.5M, $2,500.75, and 3000 dollars on 03/15/2025.",
    
    # Account numbers embedded in text
    "Please transfer funds to checking account 9876543210 before the deadline 12/31/2025.",
    
    # SSN with different separators
    "His Social Security Number is 987.65.4321, and his backup SSN is 123 45 6789.",
    
    # Form types mixed with numbers
    "Submit Forms 1099, W-2, and 1040 by April 15th and May 31st.",
    
    # Consecutive dates
    "The payment dates are March 1, 2024, March 15, 2024, and April 1, 2024.",
    
    # Multiple amounts and overlapping entities
    "The invoice shows $1,200 for equipment and $300 for shipping, sent to Amazon Inc.",
    
    # Location edge cases
    "Jane Doe moved from Los Angeles to San Francisco on May 5, 2023.",
    
    # Mixing numbers with letters (should not be ACCOUNT)
    "Invoice #12345A needs to be processed by June 30th."
]


# -------------------------------
# Run batch inference
# -------------------------------
batch_results = batch_ner(test_sentences)

# -------------------------------
# Print results in a table
# -------------------------------
for sentence, entities in batch_results:
    print("\nSentence:", sentence)
    if entities:
        table = tabulate(entities, headers=["Entity", "Type"], tablefmt="fancy_grid")
        print(table)
    else:
        print("No entities detected.")



Sentence: John Smith from New York deposited $5,000 into account 123456789 at Bank of America on March 15, 2024. His Social Security Number 123-45-6789 was used to file Form 1040.
╒═════════════════╤═════════╕
│ Entity          │ Type    │
╞═════════════════╪═════════╡
│ john smith      │ PER     │
├─────────────────┼─────────┤
│ new york        │ ORG     │
├─────────────────┼─────────┤
│ $ 5 , 000       │ AMOUNT  │
├─────────────────┼─────────┤
│ 123456789       │ ACCOUNT │
├─────────────────┼─────────┤
│ bank of america │ ORG     │
├─────────────────┼─────────┤
│ march 15 , 2024 │ DATE    │
├─────────────────┼─────────┤
│ 123 - 45 - 6789 │ SSN     │
├─────────────────┼─────────┤
│ form 1040       │ FORM    │
╘═════════════════╧═════════╛

Sentence: Alice Johnson and Bob Lee joined Microsoft Corporation and Google LLC respectively on January 2, 2023.
╒═══════════════════════╤════════╕
│ Entity                │ Type   │
╞═══════════════════════╪════════╡
│ alice johnson         │ PER 

In [13]:
import re
import torch

# ✅ Regex patterns for stronger detection
regex_patterns = {
    "SSN": r"\b\d{3}[- ]?\d{2}[- ]?\d{4}\b",
    "ACCOUNT": r"\b\d{6,12}\b",
    "AMOUNT": r"\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?|\b\d+(\.\d+)?\s?(dollars|USD|M|K|%)\b",
    "DATE": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s?\d{1,2},?\s?\d{2,4})\b",
    "FORM": r"\b(1040|W-2|1099|1098|I-9|SS-5|[Ff]orm\s?\d{3,4}[A-Z]?)\b"
}

# ✅ Label mapping
extended_id2label = {
    0: 'O',
    1: 'PER_B', 2: 'PER_I',
    3: 'LOC_B', 4: 'LOC_I',
    5: 'ORG_B', 6: 'ORG_I',
    7: 'AMOUNT_B', 8: 'AMOUNT_I',
    9: 'DATE_B', 10: 'DATE_I',
    11: 'ACCOUNT_B', 12: 'ACCOUNT_I',
    13: 'SSN_B', 14: 'SSN_I',
    15: 'FORM_B', 16: 'FORM_I'
}

# ✅ Wrapper around NER pipeline
def clean_ner_output(model, tokenizer, sentence, device="mps"):
    # Tokenize
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs).logits
    preds = torch.argmax(outputs, dim=2)[0].cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [extended_id2label[p] for p in preds]

    # Extract model entities
    entities = []
    current_entity = []
    current_label = None

    for token, label in zip(tokens, labels):
        if label.endswith("_B"):
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
            current_entity = [token.replace("##", "")]
            current_label = label.split("_")[0]
        elif label.endswith("_I") and current_entity:
            current_entity.append(token.replace("##", ""))
        else:
            if current_entity:
                entities.append((" ".join(current_entity), current_label))
                current_entity = []
                current_label = None

    if current_entity:
        entities.append((" ".join(current_entity), current_label))

    # ✅ Regex correction layer
    corrected_entities = []
    for text, label in entities:
        fixed_label = label
        for key, pattern in regex_patterns.items():
            if re.search(pattern, text, flags=re.IGNORECASE):
                fixed_label = key.upper()
                break
        corrected_entities.append((text, fixed_label))

    return corrected_entities

# -------------------------
# ✅ Example usage
# -------------------------
def test_sentences(model, tokenizer):
    sents = [
        "John Smith deposited $5000 into account 123456789 on March 15, 2024.",
        "Form 1040 must be filed by April 15th, 2023.",
        "Jane Doe with SSN 123-45-6789 received 20% bonus.",
        "Send payment of 2.5 million USD to routing number 987654321."
    ]

    for s in sents:
        ents = clean_ner_output(model, tokenizer, s, device="mps")
        print(f"Sentence: {s}")
        print(f"Detected Entities: {ents}\n")

test_sentences(extended_model, extended_tokenizer)

Sentence: John Smith deposited $5000 into account 123456789 on March 15, 2024.
Detected Entities: [('john smith', 'PER'), ('$ 5000', 'AMOUNT'), ('123 45 6 7 8 9', 'AMOUNT'), ('march 15 , 202 4', 'AMOUNT')]

Sentence: Form 1040 must be filed by April 15th, 2023.
Detected Entities: [('form 104 0', 'AMOUNT'), ('april 15th , 202 3', 'AMOUNT')]

Sentence: Jane Doe with SSN 123-45-6789 received 20% bonus.
Detected Entities: [('jane doe', 'PER'), ('ss', 'FORM')]

Sentence: Send payment of 2.5 million USD to routing number 987654321.
Detected Entities: [('2 . 5 million usd', 'AMOUNT'), ('98 7 65 43 21', 'AMOUNT')]

